# 11 - Prueba de Idempotencia

Trabajo Práctico - Pipeline ETL end-to-end (Bronze → Silver / DQL → Gold)

Este notebook **demuestra** (no solo afirma) que el pipeline es idempotente. Ejecuta el pipeline completo varias veces con `10_Orquestador_Pipeline` y compara los resultados guardados en `datos_empresas.auditoria.de_pipeline_ejecuciones`.

| Prueba | Qué se espera | Requisito del enunciado |
|---|---|---|
| **A. Re-ejecución sobre el mismo Bronze** | La corrida 2 tiene las mismas filas y el mismo checksum que la corrida 1, con **0 insertadas, 0 actualizadas y 0 eliminadas** en todas las tablas | No duplica registros; resultados consistentes |
| **B. DQL no acumula** | Los rechazos de la corrida 2 son los mismos que los de la corrida 1 y conservan su `_fecha_rechazo` original | Los rechazados no se acumulan |
| **C. Actualización según la estrategia** | Se agrega a Bronze una versión nueva de una empresa: Silver **actualiza** esa fila (no inserta otra), la versión vieja pasa a DQL como `duplicado_version_anterior`, y al retirarla todo vuelve al estado original | Las actualizaciones siguen la estrategia MERGE |

> Esta prueba ejecuta el pipeline 4 veces y puede tardar varios minutos. La prueba C agrega una fila temporal a Bronze RUES, marcada con `_source = 'PRUEBA_IDEMPOTENCIA'`, y **siempre la elimina al final**, aunque algo falle.

In [ ]:
%run ./00_Utilidades_Pipeline

In [ ]:
%python
from datetime import datetime

dbutils.widgets.dropdown("ejecutar_prueba_actualizacion", "si", ["si", "no"], "Ejecutar prueba C")
EJECUTAR_PRUEBA_C = dbutils.widgets.get("ejecutar_prueba_actualizacion") == "si"

PREFIJO = f"idempotencia_{datetime.now():%Y%m%d_%H%M%S}"
MARCA_PRUEBA = "PRUEBA_IDEMPOTENCIA"


def correr_pipeline(sufijo: str) -> str:
    id_ejecucion = f"{PREFIJO}_{sufijo}"
    print(f"Ejecutando pipeline completo: {id_ejecucion}")
    dbutils.notebook.run("./10_Orquestador_Pipeline", 7200, {"id_ejecucion": id_ejecucion})
    return id_ejecucion


def auditoria(id_ejecucion: str):
    return spark.table(AUDITORIA_EJECUCIONES).filter(F.col("id_ejecucion") == id_ejecucion)


def comparar(id_a: str, id_b: str):
    """Compara dos ejecuciones tabla por tabla."""
    a = auditoria(id_a).select("capa", "tabla", F.col("filas_tabla").alias("filas_a"), F.col("checksum").alias("checksum_a"))
    b = auditoria(id_b).select("tabla", F.col("filas_tabla").alias("filas_b"), F.col("checksum").alias("checksum_b"),
                               "filas_insertadas", "filas_actualizadas", "filas_eliminadas", "filas_duplicadas")
    return (a.join(b, "tabla")
             .withColumn("mismas_filas", F.col("filas_a") == F.col("filas_b"))
             .withColumn("mismo_checksum", F.col("checksum_a").eqNullSafe(F.col("checksum_b")))
             .orderBy(F.expr("CASE capa WHEN 'bronze' THEN 1 WHEN 'silver' THEN 2 WHEN 'dql' THEN 3 ELSE 4 END"), "tabla"))

---
## Prueba A: dos ejecuciones seguidas sobre el mismo Bronze

In [ ]:
%python
id_1 = correr_pipeline("1")

# Fechas de proceso / rechazo después de la corrida 1 (para la prueba B)
fechas_1 = {
    t: spark.table(t).agg(F.max("_processed_at" if capa == "silver" else "_fecha_rechazo")).collect()[0][0]
    for capa in ("silver", "dql") for t in [f[capa] for f in FUENTES.values()]
}

id_2 = correr_pipeline("2")

In [ ]:
%python
df_ab = comparar(id_1, id_2)
display(df_ab)

filas_ab = df_ab.collect()
assert all(r["mismas_filas"] and r["mismo_checksum"] for r in filas_ab), "La corrida 2 no reprodujo la corrida 1"
assert all((r["filas_duplicadas"] or 0) == 0 for r in filas_ab), "Hay llaves duplicadas"
assert all((r["filas_insertadas"] or 0) == 0 and (r["filas_actualizadas"] or 0) == 0 and (r["filas_eliminadas"] or 0) == 0
           for r in filas_ab if r["capa"] != "bronze"), "La corrida 2 modificó filas"
print("✔ PRUEBA A OK: misma cantidad de filas, mismo checksum, 0 insertadas / 0 actualizadas / 0 eliminadas.")

---
## Prueba B: DQL no acumula rechazos y conserva la fecha original

In [ ]:
%python
fechas_2 = {
    t: spark.table(t).agg(F.max("_processed_at" if capa == "silver" else "_fecha_rechazo")).collect()[0][0]
    for capa in ("silver", "dql") for t in [f[capa] for f in FUENTES.values()]
}
display(spark.createDataFrame(
    [(t, str(fechas_1[t]), str(fechas_2[t]), fechas_1[t] == fechas_2[t]) for t in fechas_1],
    "tabla string, ultima_fecha_corrida_1 string, ultima_fecha_corrida_2 string, sin_reescritura boolean"))

assert all(fechas_1[t] == fechas_2[t] for t in fechas_1), "La corrida 2 reescribió filas"
print("✔ PRUEBA B OK: la corrida 2 no reescribió ni volvió a insertar ningún registro en Silver ni en DQL.")

In [ ]:
%sql
-- Historial Delta de una tabla DQL: la última operación (corrida 2) es un MERGE sin filas afectadas
DESCRIBE HISTORY datos_empresas.dql.de_registromercantil_dql LIMIT 3;

---
## Prueba C: una actualización real se aplica según la estrategia MERGE

1. Se toma una empresa válida de Silver y se agrega a Bronze **una versión nueva**, con `razon_social` modificada y una `fecha_actualizacion` más reciente. Así funciona la ingesta incremental del TP1.
2. Corrida 3: Silver debe **actualizar** esa empresa (1 actualizada, 0 insertadas) y la versión anterior debe pasar a DQL como `duplicado_version_anterior`.
3. Se retira la fila de prueba de Bronze. En la corrida 4 todo debe volver exactamente al estado de la corrida 1 (mismo checksum).

In [ ]:
%python
if EJECUTAR_PRUEBA_C:
    empresa = (spark.table(SILVER_RUES).orderBy("codigo_camara", "matricula")
               .select("codigo_camara", "matricula", "razon_social").first())
    print(f"Empresa de prueba: cámara {empresa['codigo_camara']}, matrícula {empresa['matricula']} - {empresa['razon_social']}")

    version_nueva = (
        spark.table(BRONZE_RUES)
        .filter((F.trim("codigo_camara") == empresa["codigo_camara"]) & (F.trim("matricula") == empresa["matricula"]))
        .limit(1)
        .withColumn("razon_social", F.concat(F.col("razon_social"), F.lit(" (VERSION PRUEBA)")))
        .withColumn("fecha_actualizacion", F.concat(F.date_format(F.current_timestamp(), "yyyy/MM/dd HH:mm:ss.SSS"), F.lit("000000")))
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source", F.lit(MARCA_PRUEBA))
    )

    try:
        version_nueva.write.format("delta").mode("append").saveAsTable(BRONZE_RUES)
        id_3 = correr_pipeline("3_con_actualizacion")

        actual = (spark.table(SILVER_RUES)
                  .filter((F.col("codigo_camara") == empresa["codigo_camara"]) & (F.col("matricula") == empresa["matricula"]))
                  .select("razon_social").collect())
        display(auditoria(id_3).filter(F.col("tabla").isin(SILVER_RUES, DQL_RUES, GOLD_DIM_EMPRESAS))
                .select("tabla", "filas_tabla", "filas_insertadas", "filas_actualizadas", "filas_eliminadas"))
        display(spark.table(DQL_RUES)
                .filter((F.trim("codigo_camara") == empresa["codigo_camara"]) & (F.trim("matricula") == empresa["matricula"]))
                .select("_motivo_rechazo", "razon_social", "fecha_actualizacion", "_source", "_fecha_rechazo"))

        m3 = {r["tabla"]: r for r in auditoria(id_3).collect()}
        assert len(actual) == 1 and actual[0]["razon_social"].endswith("(VERSION PRUEBA)"), "Silver no tomó la versión nueva"
        assert m3[SILVER_RUES]["filas_actualizadas"] == 1 and m3[SILVER_RUES]["filas_insertadas"] == 0, "Silver no actualizó por llave"
        assert m3[DQL_RUES]["filas_insertadas"] == 1, "La versión anterior no llegó a DQL"
        print("✔ Corrida 3: Silver actualizó 1 fila (sin insertar) y la versión anterior quedó en DQL.")
    finally:
        spark.sql(f"DELETE FROM {BRONZE_RUES} WHERE _source = '{MARCA_PRUEBA}'")
        print("Fila de prueba eliminada de Bronze.")

    id_4 = correr_pipeline("4_restaurado")
    df_14 = comparar(id_1, id_4)
    display(df_14)
    assert all(r["mismas_filas"] and r["mismo_checksum"] for r in df_14.collect()), "El pipeline no volvió al estado original"
    print("✔ PRUEBA C OK: la actualización se aplicó por llave y, al retirarla, todas las tablas volvieron al estado de la corrida 1.")
else:
    print("Prueba C omitida (ejecutar_prueba_actualizacion = no).")

---
## Resumen de todas las ejecuciones de la prueba

In [ ]:
%python
display(spark.sql(f"""
    SELECT id_ejecucion, capa, tabla, filas_tabla, filas_insertadas, filas_actualizadas, filas_eliminadas,
           filas_duplicadas, checksum, validacion_ok
    FROM {AUDITORIA_EJECUCIONES}
    WHERE id_ejecucion LIKE '{PREFIJO}%'
    ORDER BY tabla, id_ejecucion
"""))